# Task 3: Appliance-Disaggregation Model (NILM)

**Objective**: Estimate washing-machine active power (Watts) from household aggregate smart meter readings at 1-minute resolution using the official cleaned REFIT data.

### Experimental Design & Leakage Prevention:
1. **Zero-Leakage Temporal Split**: Strict chronological split on House 2:
   - **Training Set**: 2013-09-17 to 2015-02-06 (536,071 minutes / ~17 months)
   - **Test Set**: 2015-02-06 to 2015-05-28 (133,973 minutes / ~3.5 months)
   - Shuffling was strictly avoided to prevent temporal correlation and window overlap leakage.
2. **Feature Engineering**: 54 time-series context features:
   - Bidirectional lags and leads ($\pm 1, 2, 3, 5, 10, 15, 20, 30$ minutes)
   - Rolling statistics (mean, std, max, min over 5, 15, 30, and 60 minutes)
   - Time-of-day and day-of-week cyclical encodings
3. **Models Evaluated**:
   - Always-Off Baseline (0 Watts)
   - Usage-Profile / Time-of-Day Baseline
   - Gradient Boosted NILM Regressor

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

## 1. Model Benchmark Results Comparison
Metrics evaluate continuous regression accuracy (MAE, RMSE), energy billing error (SAE), and active state detection (Precision, Recall, F1).

In [ ]:
results_df = pd.read_csv('../outputs/tables/model_performance_comparison.csv')
results_df

### Key Metric Interpretations:
- **MAE vs. RMSE Paradox**: Always-off achieves a deceptively low MAE (19.91 W) purely because washing machines are inactive >95% of the time. However, its RMSE is terrible (181.28 W), and it fails to detect a single active cycle (F1 = 0, SAE = 100% error).
- **Gradient Boosted NILM**: Slashes RMSE from 181.28 W to **131.85 W**, detects **64.5% of active wash cycles** (F1 = 0.4239), and accurately estimates total energy consumption within **11.7%** (49.68 kWh predicted vs. 44.46 kWh actual over 3.5 months of unseen smart meter data).

## 2. Actual vs. Predicted Visualizations
Displays a multi-day evaluation window comparing the aggregate mains power, ground-truth washing machine submeter, and the model's disaggregated predictions.

In [ ]:
display(Image(filename='../outputs/figures/disaggregation_actual_vs_predicted.png'))

## 3. Failure Cases & Model Limitations
1. **Signature Overlap**: Kettles, electric showers, and ovens also draw 2,000–3,000 W. When another high-power load triggers, the model can occasionally produce false-positive spikes if surrounding demand resembles wash agitation.
2. **Low-Power Agitation Under-prediction**: During low-power gentle tumbling (100–250 W) without water heating, the washing machine's load is easily drowned out by unmetered household baseload noise (TVs, PCs, refrigeration).